In [18]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# 1. Données et découpage (même graine qu'avant : mêmes jeux)
df = pd.read_csv("../data/raw/aug_train.csv")
X = df.drop(columns=["target", "enrollee_id"])
y = df["target"].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 2. Conversions
def convertir_experience(s):
    return pd.to_numeric(s.replace({"<1": "0", ">20": "21"}))

def convertir_last_new_job(s):
    return pd.to_numeric(s.replace({"never": "0", ">4": "5"}))

def convertir_numeriques(df_):
    df_ = df_.copy()
    df_["experience"] = convertir_experience(df_["experience"])
    df_["last_new_job"] = convertir_last_new_job(df_["last_new_job"])
    return df_

# 3. Pipeline de préparation
ohe = lambda: OneHotEncoder(handle_unknown="ignore", sparse_output=False)

preprocess = ColumnTransformer([
    ("num_mediane", Pipeline([
        ("conversion", FunctionTransformer(convertir_numeriques, feature_names_out="one-to-one")),
        ("imputation", SimpleImputer(strategy="median"))]), ["experience", "last_new_job"]),
    ("cat_inconnu", Pipeline([
        ("imputation", SimpleImputer(strategy="constant", fill_value="Inconnu")),
        ("encodage", ohe())]), ["company_size", "company_type", "gender", "education_level"]),
    ("cat_na", Pipeline([
        ("imputation", SimpleImputer(strategy="constant", fill_value="Non applicable")),
        ("encodage", ohe())]), ["major_discipline"]),
    ("cat_freq", Pipeline([
        ("imputation", SimpleImputer(strategy="most_frequent")),
        ("encodage", ohe())]), ["enrolled_university"]),
    ("echelle", StandardScaler(), ["city_development_index", "training_hours"]),
    ("binaire", ohe(), ["relevent_experience"]),
    ("ville", ohe(), ["city"]),
])

# 4. Le modèle final, entraîné sur le TRAIN seulement
modele_final = Pipeline([
    ("prep", preprocess),
    ("modele", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])
modele_final.fit(X_train, y_train)

print(X_train.shape, X_test.shape)   # (15326, 12) (3832, 12)

(15326, 12) (3832, 12)


In [19]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)

pred_test = modele_final.predict(X_test)

print(confusion_matrix(y_test, pred_test))
print("accuracy ", round(accuracy_score(y_test, pred_test), 3))
print("précision", round(precision_score(y_test, pred_test), 3))
print("rappel   ", round(recall_score(y_test, pred_test), 3))
print("F1       ", round(f1_score(y_test, pred_test), 3))

[[2214  663]
 [ 207  748]]
accuracy  0.773
précision 0.53
rappel    0.783
F1        0.632


In [20]:
from sklearn.metrics import roc_auc_score, average_precision_score

proba_test = modele_final.predict_proba(X_test)[:, 1]
print(round(roc_auc_score(y_test, proba_test), 3),
      round(average_precision_score(y_test, proba_test), 3))

0.81 0.555


In [21]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
proba_cv = cross_val_predict(modele_final, X_train, y_train, cv=cv,
                             method="predict_proba")[:, 1]

for seuil in [0.3, 0.4, 0.5, 0.6, 0.7]:
    p = (proba_cv >= seuil).astype(int)
    print(seuil, round(precision_score(y_train, p), 3),
          round(recall_score(y_train, p), 3), round(f1_score(y_train, p), 3))

0.3 0.366 0.87 0.515
0.4 0.455 0.794 0.578
0.5 0.52 0.741 0.611
0.6 0.567 0.66 0.61
0.7 0.586 0.503 0.541


In [22]:
noms = modele_final.named_steps["prep"].get_feature_names_out()
coefs = modele_final.named_steps["modele"].coef_[0]

tab = pd.DataFrame({"colonne": noms, "coef": coefs}).sort_values("coef")
print(tab.head(10))
print(tab.tail(10))

                                         colonne      coef
138                           ville__city_city_7 -1.668764
117                          ville__city_city_39 -1.021689
27   cat_inconnu__education_level_Primary School -0.982016
49                          ville__city_city_105 -0.978353
146                          ville__city_city_77 -0.871069
103                           ville__city_city_2 -0.835145
134                          ville__city_city_64 -0.829416
58                          ville__city_city_117 -0.728592
116                          ville__city_city_37 -0.695489
73                          ville__city_city_138 -0.633415
                               colonne      coef
155                ville__city_city_89  0.746252
94                ville__city_city_167  0.747487
156                 ville__city_city_9  0.758038
159                ville__city_city_93  0.811023
79                ville__city_city_144  0.818538
120                ville__city_city_42  0.854825
92      

In [23]:
tab["effectif_ville"] = np.nan
villes = tab["colonne"].str.startswith("ville__city_")
tab_sans_ville = tab[~villes]
print(tab_sans_ville.head(8))
print(tab_sans_ville.tail(8))

extremes = ["city_155", "city_21", "city_7", "city_39", "city_105"]
print(X_train["city"].value_counts().reindex(extremes))

                                        colonne      coef  effectif_ville
27  cat_inconnu__education_level_Primary School -0.982016             NaN
38              echelle__city_development_index -0.611681             NaN
12     cat_inconnu__company_type_Funded Startup -0.473946             NaN
23     cat_inconnu__education_level_High School -0.424462             NaN
24         cat_inconnu__education_level_Inconnu -0.397605             NaN
3             cat_inconnu__company_size_100-500 -0.268777             NaN
7             cat_inconnu__company_size_500-999 -0.268052             NaN
37  cat_freq__enrolled_university_no_enrollment -0.245304             NaN
                                    colonne      coef  effectif_ville
2           cat_inconnu__company_size_10/49  0.131261             NaN
16  cat_inconnu__company_type_Public Sector  0.192747             NaN
13        cat_inconnu__company_type_Inconnu  0.202938             NaN
15          cat_inconnu__company_type_Other  0.261154 